In [1]:
from useful_functions import format_datetime64, get_daterange_str

from datetime import datetime
import xarray as xr

data_out_dir = '../data_out/'

In [2]:
def update_var_attr(da, long_name=None, standard_name=None, units=None, comments=None, temporal_res=None):
    if long_name is not None:
        da.attrs['long_name'] = long_name
    if standard_name is not None:
        da.attrs['standard_name'] = standard_name
    if units is not None:
        da.attrs['units'] = units
    if comments is not None:
        da.attrs['comments'] = comments
    if temporal_res is not None:
        da.attrs['temporal_resolution'] = temporal_res

## Script settings

In [3]:
rhi_angle = 25  # azimuth angle of RHI scan in degree
rgl = 18        # range gate length in m

icon_exp = 'v370_2030'

## Load data

In [4]:
ds_icon_in = xr.open_dataset(f'/Users/geoalxx/Python/glacier_space/data_icon_h3/levante_{icon_exp}/ICON_Slice_rv_{rhi_angle}_deg.nc')
ds_rhi_in = xr.open_dataset(f'/Users/geoalxx/Python/glacier_space/data_obs_h3/streamline/RHI_dv_{rhi_angle}_deg_rgl_{rgl}.nc')

In [5]:
# Print meta data
print('\nFile Configuration:')
for k in ['Range Gate Length','View Angle','errdv_filtered']:
    print('-', k, ':', ds_rhi_in.attrs[k])

lon, lat, elev = float(ds_rhi_in.attrs['Longitude']), float(ds_rhi_in.attrs['Latitude']), float(ds_rhi_in.attrs['Altitude'])
elev = 2720


File Configuration:
- Range Gate Length : 18
- View Angle : 25
- errdv_filtered : 0.1


## Build datasets

### ICON

In [6]:
ds_icon_out = ds_icon_in.copy(deep=True)

# Update global attributes
ds_icon_out.attrs = {
    "title": f"Radial velocity for wind vectors from ICON-LES for HEFEX III campaign at Lidar location ({rhi_angle}° angle RHI scan)",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"icon-2024.10 (https://gitlab.dkrz.de/icon/icon-model.git)",
    "history": f"Created {datetime.now().strftime('%Y-%m-%d')}",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "experiment_id": f"hefex3/{icon_exp}/exp_R3B15_51m",
    "StartTime": format_datetime64( ds_icon_out.time.values[0] ),
    "StopTime":  format_datetime64( ds_icon_out.time.values[-1] ),
    "comment": f"30min averaged u,v and w values of ICON-LES simulation have been projected to line of sight ({rhi_angle}° angle) of the doppler lidar in order to compute radial velocity; Simulation based on glacier outlines from OGGM SSP370, year 2030"
}

# Update variable attributes
t_res = '30min averages (origin is the last value of the timeseries)'
update_var_attr(ds_icon_out.rv,
                long_name='radial velocity from wind vector',
                standard_name='radial_velocity',
                units='m s-1',
                comments='Radial velocity computed by projecting the 3D wind vector from ICON-LES onto the line of sight of the doppler lidar; positive values imply movement away from the instrument',
                temporal_res=t_res)

# Write
output_file = data_out_dir + f'icon/HEFEX3__ICON_{icon_exp}_lidar_location_{rhi_angle}deg__avg30min_{get_daterange_str(ds_icon_out)}.nc'
ds_icon_out.to_netcdf(output_file)
ds_icon_out.close()
print(f'Output file saved as {output_file}')

Output file saved as ../data_out/icon/HEFEX3__ICON_v370_2030_lidar_location_25deg__avg30min_20250806-20250831.nc


### RHI Scans

In [7]:
ds_rhi_out = ds_rhi_in.copy(deep=True)

# Update global attributes
upd_attr = {
    "title": f"RHI scans ({rhi_angle}° angle) from HALO StreamLine doppler lidar acquired during HEFEX III campaign",
    "institution": "Humboldt-Universität zu Berlin",
    "source": f"HALO StreamLine doppler lidar; processed with dl_toolbox (https://github.com/mkay-atm/dl_toolbox); see file configuration for details on processing settings",
    "instrument_mode": "RHI scan",
    "contact": "alexander.georgi.1@geo.hu-berlin.de (ORCID: 0009-0000-9465-6761)",
    "campaign": "HEFEX III",
    "location": f"HALO Streamline (lon, lat, elev): {lon:.6f}°E, {lat:.6f}°N, {elev:.2f}m",
    "StartTime": format_datetime64( ds_rhi_out.half_hour.values[0] ),
    "StopTime":  format_datetime64( ds_rhi_out.half_hour.values[-1] ),
    "comment": f"Manual post-processing includes 1° resampling, 30min averaging, computation of Cartesian coordinates x and z; Raw data available on request"
}

# Remove global attributes
for k in ['Title','Altitude']:
    if k in ds_rhi_out.attrs.keys():
        del ds_rhi_out.attrs[k]

ds_rhi_out.attrs = {**upd_attr, **ds_rhi_out.attrs}

# Update variable attributes
t_res = '30min averages (origin is the last value of the timeseries)'
update_var_attr(ds_rhi_out.dv,
                long_name='radial velocity of scatterers away from instrument',
                standard_name='doppler_velocity',
                units='m s-1',
                comments='A velocity is a vector quantity; the component of the velocity of the scatterers along the line of sight of the instrument where positive implies movement away from the instrument',
                temporal_res=t_res)

update_var_attr(ds_rhi_out.errdv,
                long_name='error of Doppler velocity',
                standard_name='doppler_velocity_error',
                units='m s-1',
                comments='error of radial velocity calculated from Cramer-Rao lower bound (CRLB)',
                temporal_res=t_res)

update_var_attr(ds_rhi_out.range,
                long_name='line of sight distance towards the center of each range gate',
                units='m')

update_var_attr(ds_rhi_out.zenith_bins,
                long_name='bin of beam direction due zenith',
                comments='resampling of zenith coordinate to 1-degree bins in order to make temporal averaging over spatial coordinates possible')

update_var_attr(ds_rhi_out.x,
                long_name='Cartesian coordinate x',
                comments='Computation of additional Cartesian coordinates x and z from zenith and range')

update_var_attr(ds_rhi_out.z,
                long_name='Cartesian coordinate z',
                comments='Computation of additional Cartesian coordinates x and z from zenith and range')

# Write
output_file = data_out_dir + f'obs/HEFEX3__Obs_RHI_{rhi_angle}deg_RGL{rgl}m_L2__avg30min_{get_daterange_str(ds_rhi_out, time_dim='half_hour')}.nc'
ds_rhi_out.to_netcdf(output_file)
ds_rhi_out.close()
print(f'Output file saved as {output_file}')

Output file saved as ../data_out/obs/HEFEX3__Obs_RHI_25deg_RGL18m_L2__avg30min_20250806-20250817.nc
